# MindLens Distortion Classifier Training

Kaggle T4 x2 notebook for training the MindLens cognitive distortion classifier.


In [ ]:
!pip install -q transformers datasets accelerate huggingface_hub scikit-learn

In [ ]:
from huggingface_hub import login, HfApi
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged into HuggingFace")

In [ ]:
import json
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from datasets import load_from_disk
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

LABELS = [
    "catastrophizing",
    "mind_reading",
    "all_or_nothing",
    "personalization",
    "overgeneralization",
    "emotional_reasoning",
    "should_statements",
    "jumping_to_conclusions",
    "magnification",
    "mental_filter",
]

NUM_LABELS = len(LABELS)
MODEL_NAME = "roberta-base"
YOUR_HF_USERNAME = "AmiruMallawarachchi"
OUTPUT_DIR = Path("/kaggle/working/mindlens-distortion-classifier")
DATA_PATH = Path("/kaggle/input/counselchat-distortion-cleaned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
class DistortionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float32),
        }

def extract_labels(example: dict[str, Any]) -> list[int]:
    if "labels" in example and isinstance(example["labels"], list):
        raw = example["labels"]
        if len(raw) == NUM_LABELS:
            return [int(x) for x in raw]
    if "cognitive_distortions" in example:
        vector = [0] * NUM_LABELS
        distortions = example["cognitive_distortions"]
        if isinstance(distortions, list):
            for item in distortions:
                label = str(item).strip().lower()
                if label in LABELS:
                    vector[LABELS.index(label)] = 1
        return vector
    raise ValueError(f"Cannot extract labels from example: {example}")

class MultiLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if not isinstance(labels, torch.Tensor):
            labels = torch.tensor(labels, dtype=torch.float32, device=logits.device)
        else:
            labels = labels.to(logits.device).float()
        loss_fct = nn.BCEWithLogitsLoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    probs = 1 / (1 + np.exp(-predictions))
    preds = (probs > 0.5).astype(int)
    return {
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }

In [ ]:
print(f"Loading dataset from: {DATA_PATH}")
ds = load_from_disk(str(DATA_PATH))
train_split = ds["train"]
val_split = ds["validation"] if "validation" in ds else ds["test"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_texts = [ex["text"] for ex in train_split]
train_labels = [extract_labels(ex) for ex in train_split]
val_texts = [ex["text"] for ex in val_split]
val_labels = [extract_labels(ex) for ex in val_split]

train_dataset = DistortionDataset(train_texts, train_labels, tokenizer)
val_dataset = DistortionDataset(val_texts, val_labels, tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    id2label={i: label for i, label in enumerate(LABELS)},
    label2id={label: i for i, label in enumerate(LABELS)},
)

In [ ]:
args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_strategy="epoch",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)

trainer = MultiLabelTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("\n>>> STARTING DISTORTION TRAINING <<<\n")
trainer.train()

metrics = trainer.evaluate()
print("\n" + "=" * 60)
print(f"FINAL MACRO F1: {metrics['eval_f1_macro']:.4f}")
print(f"FINAL MICRO F1: {metrics['eval_f1_micro']:.4f}")
print(f"FINAL RECALL:   {metrics['eval_recall_macro']:.4f}")
print(f"FINAL PRECISION:{metrics['eval_precision_macro']:.4f}")
print("=" * 60)

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

with open(OUTPUT_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump({
        "labels": LABELS,
        "id2label": {i: label for i, label in enumerate(LABELS)},
        "label2id": {label: i for i, label in enumerate(LABELS)},
    }, f, indent=2)

api = HfApi()
repo_id = f"{YOUR_HF_USERNAME}/mindlens-distortion-classifier"
api.create_repo(repo_id=repo_id, exist_ok=True)
api.upload_folder(folder_path=str(OUTPUT_DIR), repo_id=repo_id)
print(f"\n>>> UPLOADED: https://huggingface.co/{repo_id}")